# 🎬 SHORT MAKER - ALL-IN-ONE
Tạo video Short dạy tiếng Anh viral trong 3 bước:
1. **Bước 1**: Cài đặt hệ thống (chạy 1 lần)
2. **Bước 2**: Nhập chủ đề → Tự động sinh kịch bản + giọng đọc + phụ đề chính xác từng từ
3. **Bước 3**: Tải file zip về máy → Chạy `render_web.py` ở local

**Yêu cầu**: Kết nối GPU runtime (Runtime → Change runtime type → T4 GPU)

In [ ]:
# @title ⚙️ BƯỚC 1: CÀI ĐẶT HỆ THỐNG (Chạy 1 lần, mất ~2 phút)
import os
from IPython.display import Audio, display, clear_output

print("⏳ Đang cài đặt thư viện...")
os.system('pip install -q -U qwen-tts huggingface_hub pydub openai-whisper')
os.system('apt-get install -y -qq ffmpeg sox libsox-fmt-all')
clear_output()

from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import gc
import json
import re
import requests
import time
import zipfile
import shutil
from pathlib import Path

torch.backends.cudnn.benchmark = True
current_model = None
current_model_name = None

def load_qwen_model(model_id):
    global current_model, current_model_name
    if current_model_name == model_id and current_model is not None:
        return current_model
    if current_model is not None:
        print("🧹 Đang dọn dẹp bộ nhớ RAM...")
        del current_model
        gc.collect()
        torch.cuda.empty_cache()
    print(f"📥 Đang tải Model: {model_id.split('/')[-1]}...")
    current_model = Qwen3TTSModel.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="cuda:0",
        attn_implementation="sdpa"
    )
    current_model_name = model_id
    print("✅ Đã tải xong Model!")
    return current_model

print("✅ HỆ THỐNG ALL-IN-ONE ĐÃ SẴN SÀNG!")
print("   → Chạy tiếp Bước 2 bên dưới")

In [ ]:
# @title 🚀 BƯỚC 2: NHẬP CHỦ ĐỀ → TỰ ĐỘNG TẠO TẤT CẢ
# @markdown ---
# @markdown ### 📝 Cấu hình
topic = "How to use the word GET in English" # @param {type:"string"}
groq_api_key = "" # @param {type:"string"}
# @markdown > Lấy Groq API key miễn phí tại: https://console.groq.com/keys
# @markdown ---
# @markdown ### 🎤 Cấu hình giọng đọc
voice_style = "Chelsie" # @param ["Chelsie", "Ethan", "Aiden"]
speaking_speed = 1.0 # @param {type:"slider", min:0.7, max:1.3, step:0.05}

if not topic.strip():
    raise ValueError("❌ Hãy nhập chủ đề video!")
if not groq_api_key.strip():
    raise ValueError("❌ Hãy nhập Groq API key!")

# === 1. Tạo tên project ===
slug = re.sub(r'[^a-z0-9]+', '_', topic.lower()).strip('_')
project_dir = Path(f'/content/{slug}')
project_dir.mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f"🎬 SHORT MAKER: {topic}")
print(f"{'='*60}")

# === 2. Sinh kịch bản bằng Groq ===
print(f"\n📖 [1/4] Đang sinh kịch bản bằng AI...")

PROMPT_TEMPLATE = """You are a top-tier viral YouTube Shorts scriptwriter and English language educator. Write an engaging English learning/storytelling script about: "{topic}"

## CRITICAL LENGTH REQUIREMENT:
- The script MUST be 120-140 words long. This is non-negotiable.
- At normal speaking pace, 120-140 words = approximately 45-50 seconds of audio.
- Write in a flowing, storytelling narrative — NOT a bullet-point list.
- Use clear, professional, yet conversational English.

## PUNCTUATION RULES (TTS system reads these as timing cues):
- COMMAS (,) = chain actions smoothly in one breath, no pause.
- PERIODS (.) QUESTION MARKS (?) EXCLAMATION (!) = end of sentence, brief pause.
- BLANK LINES between paragraphs = dramatic 0.35s pause for emphasis.
- NEVER use ellipsis (...).

## SCRIPT STRUCTURE:
- Paragraph 1 (HOOK): Attention-grabbing question or shocking fact. 1-2 sentences max.
- Paragraphs 2-4 (BODY): Educational, informative content with key vocabulary.
- Final paragraph (CTA): Short call to action.

## VISUAL KEYWORDS:
Extract exactly 10-12 concrete nouns, actions, or phrases from YOUR script, in strict order of appearance.
- "keyword": 1-3 words from the script.
- "search_query": 3-6 word descriptive phrase for Pexels search.

## OUTPUT (valid JSON only, no markdown, no explanation):
{{
  "script": "Full script here with blank lines between paragraphs.",
  "visual_keywords": [
    {{"keyword": "alarm", "search_query": "alarm clock morning dark bedroom"}}
  ]
}}"""

prompt = PROMPT_TEMPLATE.format(topic=topic)
resp = requests.post(
    "https://api.groq.com/openai/v1/chat/completions",
    headers={"Authorization": f"Bearer {groq_api_key}", "Content-Type": "application/json"},
    json={
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {"role": "system", "content": "You are a JSON-only response bot. Always return valid JSON."},
            {"role": "user", "content": prompt}
        ],
        "response_format": {"type": "json_object"},
        "temperature": 0.8,
        "max_tokens": 2000
    },
    timeout=30
)
resp.raise_for_status()
raw = resp.json()["choices"][0]["message"]["content"].strip()
raw = re.sub(r'^```json\s*', '', raw)
raw = re.sub(r'\s*```$', '', raw)
script_data = json.loads(raw)

script_text = script_data["script"]
keywords = script_data.get("visual_keywords", [])

# Lưu file
(project_dir / "script.txt").write_text(script_text, encoding="utf-8")
(project_dir / "keywords.json").write_text(json.dumps(script_data, indent=2, ensure_ascii=False), encoding="utf-8")

word_count = len(script_text.split())
print(f"   ✅ Kịch bản: {word_count} từ, {len(keywords)} visual keywords")
print(f"   📄 Script:")
print(f"   {'-'*50}")
for line in script_text.split('\n'):
    print(f"   {line}")
print(f"   {'-'*50}")

# === 3. Tạo giọng đọc bằng Qwen3-TTS ===
print(f"\n🎤 [2/4] Đang tạo giọng đọc ({voice_style})...")

model = load_qwen_model("Qwen/Qwen3-TTS")

spk_text = f"[{voice_style}]: {script_text}"

audio_path = str(project_dir / "audio.wav")
wav = model.synthesize(
    spk_text,
    speed=speaking_speed
)
sf.write(audio_path, wav, samplerate=24000)

# Convert WAV → MP3
mp3_path = str(project_dir / "audio.mp3")
os.system(f'ffmpeg -y -i "{audio_path}" -codec:a libmp3lame -qscale:a 2 "{mp3_path}" -loglevel error')
os.remove(audio_path)

print(f"   ✅ Đã tạo audio.mp3")
display(Audio(mp3_path, autoplay=True))

# === 4. Tạo phụ đề chính xác từng từ bằng Whisper ===
print(f"\n📝 [3/4] Đang tạo phụ đề chính xác từng từ (Whisper)...")

import whisper

whisper_model = whisper.load_model("base.en")
result = whisper_model.transcribe(mp3_path, word_timestamps=True, language="en")

# Tạo SRT từ Whisper segments
srt_lines = []
for i, seg in enumerate(result["segments"], 1):
    start_h = int(seg["start"] // 3600)
    start_m = int((seg["start"] % 3600) // 60)
    start_s = int(seg["start"] % 60)
    start_ms = int((seg["start"] % 1) * 1000)
    end_h = int(seg["end"] // 3600)
    end_m = int((seg["end"] % 3600) // 60)
    end_s = int(seg["end"] % 60)
    end_ms = int((seg["end"] % 1) * 1000)
    srt_lines.append(f"{i}")
    srt_lines.append(f"{start_h:02d}:{start_m:02d}:{start_s:02d},{start_ms:03d} --> {end_h:02d}:{end_m:02d}:{end_s:02d},{end_ms:03d}")
    srt_lines.append(seg["text"].strip())
    srt_lines.append("")

srt_path = project_dir / "subtitle.srt"
srt_path.write_text('\n'.join(srt_lines), encoding='utf-8')

# Tạo word_timestamps.json từ Whisper (chính xác từng từ)
word_ts = []
for seg in result["segments"]:
    for w in seg.get("words", []):
        word_ts.append({
            "word": w["word"].strip(),
            "start": round(w["start"], 3),
            "end": round(w["end"], 3)
        })

ts_path = project_dir / "word_timestamps.json"
ts_path.write_text(json.dumps(word_ts, indent=2, ensure_ascii=False), encoding='utf-8')

print(f"   ✅ Đã tạo subtitle.srt ({len(result['segments'])} câu)")
print(f"   ✅ Đã tạo word_timestamps.json ({len(word_ts)} từ)")

# === 5. Đóng gói ZIP để tải về ===
print(f"\n📦 [4/4] Đóng gói ZIP...")

zip_path = f"/content/{slug}.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in project_dir.iterdir():
        if file.is_file():
            zf.write(file, f"{slug}/{file.name}")

zip_size_mb = os.path.getsize(zip_path) / (1024*1024)

print(f"\n{'='*60}")
print(f"🎉 HOÀN TẤT! ({zip_size_mb:.1f} MB)")
print(f"{'='*60}")
print(f"")
print(f"📁 File đã tạo:")
print(f"   • script.txt      — Kịch bản")
print(f"   • keywords.json   — Từ khóa hình ảnh")
print(f"   • audio.mp3       — Giọng đọc AI")
print(f"   • subtitle.srt    — Phụ đề")
print(f"   • word_timestamps.json — Sync từng từ")
print(f"")
print(f"⬇️ Chạy Bước 3 bên dưới để tải ZIP về máy")

In [ ]:
# @title 📥 BƯỚC 3: TẢI ZIP VỀ MÁY
from google.colab import files
files.download(zip_path)

print(f"")
print(f"📋 HƯỚNG DẪN TIẾP THEO:")
print(f"   1. Giải nén ZIP vào thư mục: d:\\folder\\tools\\short\\projects\\{slug}\\")
print(f"   2. (Tùy chọn) Mở extension TurboFlow trên Edge để gen ảnh AI")
print(f"   3. Mở terminal tại d:\\folder\\tools\\short và chạy:")
print(f"")
print(f"      python render_web.py {slug}")
print(f"")
print(f"   4. Video thành phẩm: projects/{slug}/output_web.mp4")